# DQN on Atari Pong: hyperparameter experiments (Kaggle)

This notebook clones the group repo, trains one DQN per experiment config with `train.py`,
scores each trained model with exploration switched off, and builds the results table and plots.

**Before you run it on Kaggle**
1. Settings → Accelerator → **GPU T4** (or P100).
2. Settings → **Internet on** (needed for `git clone` and `pip install`).
3. Set `MEMBER` and the `EXPERIMENTS` list in the config cell. Member 1 copies this notebook and swaps in their own list.
4. Do a quick interactive run of the cells up to the speed check, pick `TOTAL_STEPS`, then use
   **Save Version → Save & Run All** so training keeps going with the browser closed.

All outputs end up in `/kaggle/working` (each run's model, log, plot, config and score, plus the results table,
the comparison plot and a zip of everything) and can be downloaded from the notebook's Output tab.

## 1. GPU check

In [ ]:
import subprocess
import torch

print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Turn on the GPU accelerator in the notebook settings, otherwise training is very slow.")

## 2. Get the code and install the Atari packages
Kaggle's own torch is kept on purpose. Reinstalling torch would waste time and can break CUDA.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Samkwizera/deep_q_lerning-group-12.git"
BRANCH = "main"

if os.path.exists("train.py"):
    # running from inside a checkout of the repo (e.g. locally)
    REPO_DIR = os.getcwd()
else:
    REPO_DIR = "/tmp/deep_q_lerning-group-12"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

print("code from:", REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "gymnasium[atari]>=1.0", "ale-py>=0.10", "moviepy"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 3. Experiment config

Everyone keeps the same base settings, so the only thing that changes between runs is what the member is tuning.
`eps_decay` is given as a **fraction of `TOTAL_STEPS`**, so a config keeps its meaning if you change the run length.
The actual number of steps is written to each run's `config.json`.

Member 2 varies **epsilon start / end / decay** and keeps learning rate, gamma and batch size at the defaults.

In [ ]:
MEMBER = "member2"

# run length per experiment. Pick this after the speed check in section 4.
TOTAL_STEPS = int(os.environ.get("DQN_TOTAL_STEPS", 500_000))

# shared base settings, the same for every experiment and both members
BASE = dict(
    lr=1e-4,
    gamma=0.99,
    batch_size=32,
    buffer_size=50_000,
    learning_starts=int(os.environ.get("DQN_LEARNING_STARTS", 10_000)),
    train_freq=4,
    target_update=1_000,
    seed=42,
)

# Member 2: epsilon start / end / decay (decay = fraction of TOTAL_STEPS)
EXPERIMENTS = [
    dict(name="exp01_baseline",          eps_start=1.0, eps_end=0.05, eps_decay=0.50),
    dict(name="exp02_fast_decay",        eps_start=1.0, eps_end=0.05, eps_decay=0.10),
    dict(name="exp03_slow_decay",        eps_start=1.0, eps_end=0.05, eps_decay=0.90),
    dict(name="exp04_low_end",           eps_start=1.0, eps_end=0.01, eps_decay=0.50),
    dict(name="exp05_high_end",          eps_start=1.0, eps_end=0.20, eps_decay=0.50),
    dict(name="exp06_low_start",         eps_start=0.5, eps_end=0.05, eps_decay=0.50),
    dict(name="exp07_constant_eps",      eps_start=0.1, eps_end=0.10, eps_decay=0.50),
    dict(name="exp08_greedy_start",      eps_start=0.2, eps_end=0.01, eps_decay=0.25),
    dict(name="exp09_fast_decay_low_end", eps_start=1.0, eps_end=0.01, eps_decay=0.10),
    dict(name="exp10_slow_decay_high_end", eps_start=1.0, eps_end=0.10, eps_decay=0.75),
]

# run only some experiments in this session, e.g. ["exp01_baseline", "exp02_fast_decay"]. None runs all.
RUN_ONLY = None

# episodes used to score each trained model with epsilon = 0
EVAL_EPISODES = int(os.environ.get("DQN_EVAL_EPISODES", 5))

# outputs: /kaggle/working on Kaggle, ./kaggle_outputs elsewhere
OUT_ROOT = os.environ.get("DQN_OUT_ROOT") or ("/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("kaggle_outputs"))
RUNS_DIR = os.path.join(OUT_ROOT, "runs", MEMBER)
os.makedirs(RUNS_DIR, exist_ok=True)

# Resuming over several sessions: add the output of an earlier version of this notebook as an input
# and put its runs folder here, e.g. "/kaggle/input/<notebook-name>/runs/member2". Finished runs are copied
# over and skipped.
RESUME_FROM = None

names = [e["name"] for e in EXPERIMENTS]
assert len(names) == len(set(names)), "experiment names must be unique"
print(f"{len(EXPERIMENTS)} experiments, {TOTAL_STEPS:,} steps each, outputs in {RUNS_DIR}")

## 4. Speed check (optional)
A short run to measure steps per second on this GPU. Multiply your planned `TOTAL_STEPS` × 10 runs by it to
check you fit in the session (12 h max) and your weekly GPU quota. Set `RUN_SPEED_CHECK = False` to skip it.

In [ ]:
import time

RUN_SPEED_CHECK = os.environ.get("DQN_SPEED_CHECK", "1") == "1"

if RUN_SPEED_CHECK:
    check_steps = 20_000
    t0 = time.time()
    subprocess.run([sys.executable, os.path.join(REPO_DIR, "train.py"),
                    "--total-steps", str(check_steps), "--learning-starts", "5000",
                    "--out-dir", "/tmp/speed_check"], check=True, capture_output=True, text=True)
    sps = check_steps / (time.time() - t0)
    est_min = TOTAL_STEPS / sps / 60
    print(f"about {sps:.0f} steps/s (includes start-up, so slightly pessimistic)")
    print(f"one {TOTAL_STEPS:,}-step run: about {est_min:.0f} min; all {len(EXPERIMENTS)} runs: about {est_min * len(EXPERIMENTS) / 60:.1f} h")

## 5. Helpers: training and scoring

In [ ]:
import json
import shutil

import numpy as np
from train import QNetwork, make_env


def to_cli_args(exp):
    decay_steps = max(1, int(exp["eps_decay"] * TOTAL_STEPS))
    params = dict(BASE, eps_start=exp["eps_start"], eps_end=exp["eps_end"], eps_decay=decay_steps,
                  total_steps=TOTAL_STEPS)
    args = []
    for key, value in params.items():
        args += ["--" + key.replace("_", "-"), str(value)]
    return args


def train_run(exp, out_dir):
    cmd = [sys.executable, "-u", os.path.join(REPO_DIR, "train.py"), *to_cli_args(exp), "--out-dir", out_dir]
    print(" ".join(cmd[2:]))
    t0 = time.time()
    with open(os.path.join(out_dir, "train_stdout.txt"), "w") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            log.write(line)
            # print every 10th episode so the notebook output stays readable; the full log is in train_stdout.txt
            if not line.startswith("ep ") or int(line.split()[1]) % 10 == 0:
                print(line, end="")
        proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"train.py failed for {exp['name']}, see {out_dir}/train_stdout.txt")
    return (time.time() - t0) / 60


def evaluate(model_path, episodes=EVAL_EPISODES):
    # epsilon = 0: always the action the network rates highest, like play.py
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    env = make_env()
    q_net = QNetwork(env.action_space.n).to(device)
    q_net.load_state_dict(torch.load(model_path, map_location=device))
    q_net.eval()
    rewards, lengths = [], []
    for i in range(episodes):
        state, _ = env.reset(seed=1000 + i)
        total, steps, done = 0.0, 0, False
        while not done:
            with torch.no_grad():
                action = int(q_net(torch.as_tensor(np.asarray(state), device=device).unsqueeze(0)).argmax(dim=1).item())
            state, reward, terminated, truncated, _ = env.step(action)
            total += reward
            steps += 1
            done = terminated or truncated
        rewards.append(total)
        lengths.append(steps)
    env.close()
    return dict(eval_episodes=episodes, eval_mean=float(np.mean(rewards)), eval_std=float(np.std(rewards)),
                eval_rewards=rewards, eval_lengths=lengths)

## 6. Run the experiments

In [ ]:
if RESUME_FROM:
    for name in os.listdir(RESUME_FROM):
        src, dst = os.path.join(RESUME_FROM, name), os.path.join(RUNS_DIR, name)
        if os.path.exists(os.path.join(src, "eval.json")) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print("resumed", name)

todo = [e for e in EXPERIMENTS if RUN_ONLY is None or e["name"] in RUN_ONLY]
for i, exp in enumerate(todo, 1):
    out_dir = os.path.join(RUNS_DIR, exp["name"])
    model_path = os.path.join(out_dir, "dqn_model.pth")
    eval_path = os.path.join(out_dir, "eval.json")
    print(f"\n===== [{i}/{len(todo)}] {exp['name']} =====")
    if os.path.exists(eval_path):
        print("already done, skipping")
        continue
    os.makedirs(out_dir, exist_ok=True)
    minutes = None
    if not os.path.exists(model_path):
        minutes = train_run(exp, out_dir)
    result = evaluate(model_path)
    result.update(name=exp["name"], eps_decay_fraction=exp["eps_decay"], train_minutes=minutes)
    with open(eval_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"score with epsilon=0 over {result['eval_episodes']} episodes: "
          f"{result['eval_mean']:.1f} ± {result['eval_std']:.1f}  (trained in {minutes or 0:.0f} min)")

## 7. Results table
`behaviour` is left empty for you to fill in after watching the agents, e.g. "tracks the ball but misses fast returns".

In [ ]:
import pandas as pd

rows = []
for exp in EXPERIMENTS:
    out_dir = os.path.join(RUNS_DIR, exp["name"])
    if not os.path.exists(os.path.join(out_dir, "eval.json")):
        continue
    cfg = json.load(open(os.path.join(out_dir, "config.json")))
    ev = json.load(open(os.path.join(out_dir, "eval.json")))
    log = pd.read_csv(os.path.join(out_dir, "log.csv"))
    mean20 = log["reward"].rolling(20, min_periods=1).mean()
    rows.append(dict(
        experiment=exp["name"],
        lr=cfg["lr"], gamma=cfg["gamma"], batch_size=cfg["batch_size"],
        eps_start=cfg["eps_start"], eps_end=cfg["eps_end"], eps_decay_steps=cfg["eps_decay"],
        total_steps=cfg["total_steps"],
        episodes=len(log),
        train_reward_last20=round(mean20.iloc[-1], 2),
        train_reward_best20=round(mean20.max(), 2),
        eval_reward_mean=round(ev["eval_mean"], 2),
        eval_reward_std=round(ev["eval_std"], 2),
        train_minutes=round(ev["train_minutes"], 1) if ev.get("train_minutes") else None,
        behaviour="",
    ))

results = pd.DataFrame(rows)
results.to_csv(os.path.join(OUT_ROOT, f"{MEMBER}_results.csv"), index=False)
with open(os.path.join(OUT_ROOT, f"{MEMBER}_results.md"), "w") as f:
    try:
        f.write(results.to_markdown(index=False))
    except ImportError:  # to_markdown needs the tabulate package
        f.write(results.to_string(index=False))
results

## 8. Comparison plots

In [ ]:
import matplotlib.pyplot as plt

if not results.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw=dict(width_ratios=[2, 1]))
    for name in results["experiment"]:
        log = pd.read_csv(os.path.join(RUNS_DIR, name, "log.csv"))
        ax1.plot(log["total_steps"], log["reward"].rolling(20, min_periods=1).mean(), label=name)
    ax1.set_xlabel("total steps")
    ax1.set_ylabel("training reward (mean of last 20 episodes)")
    ax1.set_title(f"{MEMBER}: training curves")
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.3)

    ordered = results.sort_values("eval_reward_mean")
    ax2.barh(ordered["experiment"], ordered["eval_reward_mean"], xerr=ordered["eval_reward_std"], color="tab:blue")
    ax2.set_xlabel(f"score with epsilon = 0 (mean of {EVAL_EPISODES} episodes)")
    ax2.set_title("final model scores")
    ax2.grid(alpha=0.3, axis="x")

    fig.tight_layout()
    fig.savefig(os.path.join(OUT_ROOT, f"{MEMBER}_comparison.png"), dpi=120)
    plt.show()

## 9. Best and worst config
Ranked by the epsilon = 0 score, with ties broken by the training reward at the end.
The best model is copied to `<member>_best_dqn_model.pth`. The final `dqn_model.pth` for the repo is picked jointly across all 20 runs.

In [ ]:
if not results.empty:
    ranked = results.sort_values(["eval_reward_mean", "train_reward_last20"], ascending=False)
    best, worst = ranked.iloc[0], ranked.iloc[-1]
    print("best: ", best["experiment"], "| score", best["eval_reward_mean"], "| train last20", best["train_reward_last20"])
    print("worst:", worst["experiment"], "| score", worst["eval_reward_mean"], "| train last20", worst["train_reward_last20"])
    BEST_MODEL = os.path.join(OUT_ROOT, f"{MEMBER}_best_dqn_model.pth")
    shutil.copy(os.path.join(RUNS_DIR, best["experiment"], "dqn_model.pth"), BEST_MODEL)
    print("copied to", BEST_MODEL)

## 10. Gameplay video of the best model (optional)
Kaggle can't open a game window, so this records an mp4 with `play.py --record` instead. It's a backup for the live demo,
which runs `play.py` on a laptop with the downloaded model.

In [ ]:
RECORD_VIDEO = os.environ.get("DQN_RECORD_VIDEO", "1") == "1"

if RECORD_VIDEO and not results.empty:
    subprocess.run([sys.executable, os.path.join(REPO_DIR, "play.py"), "--model", BEST_MODEL,
                    "--episodes", "1", "--record"], cwd=OUT_ROOT, check=True)
    print("videos:", os.listdir(os.path.join(OUT_ROOT, "videos")))

## 11. Zip everything for download

In [ ]:
archive = shutil.make_archive(os.path.join(OUT_ROOT, f"{MEMBER}_outputs"), "zip", root_dir=OUT_ROOT,
                              base_dir=os.path.relpath(RUNS_DIR, OUT_ROOT))
for extra in [f"{MEMBER}_results.csv", f"{MEMBER}_results.md", f"{MEMBER}_comparison.png"]:
    print(extra, "exists:", os.path.exists(os.path.join(OUT_ROOT, extra)))
print("zip:", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")